# Semantic LLM Router — Simulation & Tests

**No GPU needed.** Pure Python asyncio — free Colab CPU is sufficient.

| Section | What it tests |
|---------|--------------|
| Step 1 | Setup (clone + install) |
| Step 2 | Load balancing simulation — 5 scenarios |
| Step 3 | Latency penalty demo |
| Step 4 | Accuracy penalty demo |
| Step 5 | Semantic analyzer classification |
| Step 6 | Analyzer confidence scores |

## Step 1 — Setup (safe to re-run)

In [ ]:
import os, subprocess, sys

REPO = '/content/semantic-llm-router'
URL  = 'https://github.com/yfan000/semantic-llm-router.git'

# IMPORTANT: go to a safe parent dir BEFORE removing the repo,
# otherwise rm -rf deletes our cwd and git loses its working directory.
os.chdir('/content')

if os.path.exists(REPO):
    subprocess.run(['rm', '-rf', REPO], check=True)
    print('Removed previous clone.')

print('Cloning...')
ret = subprocess.call(['git', 'clone', URL, REPO])
if ret != 0:
    raise RuntimeError(f'git clone failed (exit {ret}). Check network.')

os.chdir(REPO)
print('Working directory:', os.getcwd())
print('Files:', sorted(f for f in os.listdir('.') if not f.startswith('.')))

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'fastapi', 'pydantic', 'numpy', 'sentence-transformers'])
print('✓ Setup complete.')

## Step 2 — Load balancing simulation (5 scenarios, ~3 min total)

### Scenario 1 — Default (equal weights, Poisson arrivals)

In [ ]:
!python tests/simulation.py --scenario default --requests 200 --concurrency 10

### Scenario 2 — Spike (burst → load spike → price inflates → traffic shifts)

In [ ]:
!python tests/simulation.py --scenario spike --requests 300 --concurrency 20

### Scenario 3 — SLA pressure (min_accuracy=0.80 → 8B excluded)

In [ ]:
!python tests/simulation.py --scenario sla --requests 200 --concurrency 10

### Scenario 4 — Eco mode (energy_weight=0.40 → 8B at 26.7 tok/J dominates)

In [ ]:
!python tests/simulation.py --scenario eco --requests 200 --concurrency 10

### Scenario 5 — Mixed users (1/3 cost, 1/3 eco, 1/3 accuracy)

In [ ]:
!python tests/simulation.py --scenario mixed_users --requests 400 --concurrency 15

## Step 3 — Latency penalty demo
model-a bids 200ms but always delivers 800ms.
After 40 samples its penalty multiplies ~2.7× and it loses future auctions.

In [ ]:
import sys, tempfile
sys.path.insert(0, '/content/semantic-llm-router')
from semantic_router.reputation_tracker import ReputationTracker
from semantic_router.schemas import BidResponse, UserPreference
from semantic_router.selector import select

tracker = ReputationTracker(path=tempfile.mktemp(suffix='.json'))
def bid(mid, cost, lat, acc=0.80):
    return BidResponse(model_id=mid, estimated_cost_usd=cost, estimated_latency_ms=lat,
                       estimated_accuracy=acc, estimated_energy_j=10.0,
                       efficiency_tokens_per_joule=10.0, current_load=0.3)
pref = UserPreference(cost_weight=0.6, latency_weight=0.2, accuracy_weight=0.1, energy_weight=0.1)

print('BEFORE penalty:')
w = select([bid('model-a', 0.001, 200), bid('model-b', 0.003, 400)], pref, tracker, 'factual', 'easy')
print(f'  Winner: {w.model_id}  (model-a looks cheapest)')
for _ in range(40):
    tracker.record_latency('model-a', bid_latency_ms=200, actual_latency_ms=800)
p = tracker.get_penalty_multiplier('model-a')
print(f'\nAFTER 40 slow responses:')
print(f'  Reliability:    {tracker.get_latency_reliability("model-a"):.3f}')
print(f'  Penalty:        {p:.2f}x')
print(f'  Effective cost: ${0.001*p:.4f}  (was $0.0010)')
w = select([bid('model-a', 0.001, 200), bid('model-b', 0.003, 400)], pref, tracker, 'factual', 'easy')
print(f'  Winner: {w.model_id}  (model-b wins despite higher base cost)')

## Step 4 — Accuracy penalty demo
model-a bids accuracy=0.92 but judge scores 0.45 every time.
After 40 samples: effective accuracy = 0.92 × 0.49 ≈ 0.45 → honest model wins.

In [ ]:
import tempfile
from semantic_router.reputation_tracker import ReputationTracker
from semantic_router.schemas import BidResponse, UserPreference
from semantic_router.selector import select

tracker = ReputationTracker(path=tempfile.mktemp(suffix='.json'))
def bid(mid, cost, acc):
    return BidResponse(model_id=mid, estimated_cost_usd=cost, estimated_latency_ms=400,
                       estimated_accuracy=acc, estimated_energy_j=10.0,
                       efficiency_tokens_per_joule=10.0, current_load=0.3)
pref = UserPreference(accuracy_weight=0.60, cost_weight=0.15, latency_weight=0.15, energy_weight=0.10)

print('BEFORE accuracy penalty:')
w = select([bid('model-a', 0.001, 0.92), bid('model-b', 0.002, 0.75)], pref, tracker, 'math', 'hard')
print(f'  Winner: {w.model_id}  (model-a looks most accurate)')
for _ in range(40):
    tracker.record_accuracy_bid('model-a', 'math', 'hard', bid_accuracy=0.92, judge_score=0.45)
d = tracker.get_accuracy_discount('model-a', 'math', 'hard')
print(f'\nAFTER 40 overbidding events:')
print(f'  Discount:           {d:.3f}')
print(f'  Effective accuracy: {0.92*d:.3f}  (was 0.92)')
w = select([bid('model-a', 0.001, 0.92), bid('model-b', 0.002, 0.75)], pref, tracker, 'math', 'hard')
print(f'  Winner: {w.model_id}  (model-b wins: 0.75 > {0.92*d:.2f} effective)')

## Step 5 — Semantic analyzer classification
Tests domain + complexity classification using MiniLM-L6-v2 anchor embeddings.

In [ ]:
import sys
sys.path.insert(0, '/content/semantic-llm-router')
from semantic_router.analyzer import SemanticAnalyzer

analyzer = SemanticAnalyzer()
analyzer.load()
print('Analyzer ready.')

test_queries = [
    ('factual',   'easy',   'What is the capital of France?'),
    ('factual',   'hard',   'Explain the geopolitical causes of the 2008 financial crisis.'),
    ('math',      'easy',   'What is 15% of 200?'),
    ('math',      'hard',   'Prove that there are infinitely many prime numbers.'),
    ('code',      'easy',   'Write a Python function to reverse a string.'),
    ('code',      'hard',   'Implement a distributed rate limiter using Redis.'),
    ('reasoning', 'medium', 'Compare and contrast microservices vs monolithic architecture.'),
    ('creative',  'medium', 'Write a short story about a robot learning to paint.'),
]

print(f"{'Query':<52} {'Domain':>10} {'Complexity':>12} {'Match':>8}")
print('-' * 86)
correct = 0
for exp_domain, exp_complexity, query in test_queries:
    meta = analyzer.analyze([{'role': 'user', 'content': query}])
    d_ok = '✓' if meta.domain     == exp_domain     else '✗'
    c_ok = '✓' if meta.complexity == exp_complexity else '✗'
    ok   = '✓✓' if d_ok == '✓' and c_ok == '✓' else '✗'
    if ok == '✓✓': correct += 1
    print(f'  {query[:50]:<50}  {meta.domain:>10} {meta.complexity:>12}  [{d_ok}domain {c_ok}complexity]')
print(f'\n  Accuracy: {correct}/{len(test_queries)} correct')

## Step 6 — Analyzer confidence scores
Cosine similarity per classification. Below ~0.45 = ambiguous.

In [ ]:
queries = [
    'Solve the differential equation dy/dx = 2x',
    'Write a haiku about mountains',
    'Debug this Python function that crashes on large input',
    'What year was the Eiffel Tower built?',
    'Design a fault-tolerant microservices system for 10M users',
    'What is the probability of rolling two sixes in a row?',
]

print(f"{'Query':<52} {'Domain':>12} {'Complexity':>12} {'Dom conf':>10} {'Cplx conf':>11}")
print('-' * 102)
for q in queries:
    meta = analyzer.analyze([{'role': 'user', 'content': q}])
    conf = 'HIGH' if meta.domain_score > 0.55 else 'MED' if meta.domain_score > 0.45 else 'LOW'
    print(f'  {q[:50]:<50}  {meta.domain:>12} {meta.complexity:>12} {meta.domain_score:>10.3f} {meta.complexity_score:>11.3f}  [{conf}]')